# Second runtime: trust tests in parallel

Runtime: **L4 GPU**. 1) Mount Google Drive (Files panel, Drive icon): it restores every model session 1 trained (MyDrive/thinkingcap-mvtf). 2) Run all. The worker runs `plan_second.txt` (fresh-question tests, retrain-seed checks); it only ADDS new results and new adapter names, never overwrites session 1's. When the queue is done, the last cell releases this runtime so it stops costing units.

In [ ]:
# Google Drive first: every model session 1 trained is restored from it
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%env PYTHONUNBUFFERED=1
%env TRACK_SESSION=second
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d ThinkingCap-Test || git clone -q https://github.com/BottleCapAI/ThinkingCap-Test
%cd /content/ThinkingCap-Test
!test -f .installed || (pip -q install uv && uv pip install --system -q -r requirements.txt && touch .installed)

In [ ]:
!rm -rf /content/T && git clone -q --depth 1 -b mvt-fading https://github.com/arsenelemoigne/Test /content/T && cp /content/T/thinkingcap/*.py /content/T/thinkingcap/plan*.txt . && git -C /content/T log -1 --format='%h %s'

In [ ]:
!python track.py setup

In [ ]:
!python track.py plan --plan-file plan_second.txt --wait-hours 0.2 2>&1 | tee -a /content/track_log.txt

In [ ]:
# queue finished: release this runtime (results and models are on Drive)
from google.colab import runtime
runtime.unassign()